In [7]:
!pip install scikit-learn pandas numpy gradio nltk textblob

import pandas as pd
import numpy as np
import re
import json
import warnings
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr
import nltk
from textblob import TextBlob

warnings.filterwarnings('ignore')

In [8]:
# --- REQUISITE DATA DOWNLOADS ---
try:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
except:
    pass

class InternshipNavigator:
    def __init__(self):
        print("🛠️ System: Initializing Local Knowledge Engine...")

        # Configure the text processing engine
        self.text_engine = TfidfVectorizer(
            max_features=1200,
            stop_words='english',
            ngram_range=(1, 2),
            lowercase=True
        )

        self._load_service_data()
        self._train_similarity_model()
        print("✅ System: Navigator Ready.")

    def _load_service_data(self):
        """Standardized FAQ repository for internship services."""
        self.registry = [
            {
                "topic": "Application Protocol",
                "query": "How do I apply for an internship?",
                "info": "1) Access our portal and register. 2) Provide a PDF resume and personalized cover letter. 3) Complete the digital profile. 4) Submit before the deadline. Highlight projects that demonstrate your technical proficiency.",
                "tags": ["apply", "application", "submission", "how to", "join"]
            },
            {
                "topic": "Required Paperwork",
                "query": "What documents do I need for internship application?",
                "info": "You must provide: Updated CV (PDF), official university transcripts, a cover letter, and two academic/professional references. Portfolios are recommended for creative or technical roles.",
                "tags": ["documents", "resume", "cv", "transcript", "reference", "portfolio"]
            },
            {
                "topic": "Eligibility Standards",
                "query": "What are the eligibility requirements for internships?",
                "info": "Applicants must be enrolled in an accredited degree program (Undergrad/Postgrad), maintain a 3.0+ GPA, and possess legal work authorization. We look for a balance of academic excellence and soft skills.",
                "tags": ["eligibility", "criteria", "requirements", "gpa", "qualifications"]
            },
            {
                "topic": "Program Chronology",
                "query": "How long do internships last and what is the schedule?",
                "info": "Summer terms are 10-12 weeks; Semester terms are 14-16 weeks. We expect a 40-hour weekly commitment during breaks, while part-time roles are 20 hours/week during studies.",
                "tags": ["duration", "weeks", "months", "schedule", "hours", "long"]
            },
            {
                "topic": "Financial Benefits",
                "query": "Are internships paid and what benefits do you offer?",
                "info": "All roles are paid. Benefits include competitive hourly wages, 1-on-1 mentorship, networking sessions with executives, and potential for full-time conversion post-graduation.",
                "tags": ["paid", "salary", "stipend", "money", "benefits", "wages"]
            },
            {
                "topic": "Virtual Placement",
                "query": "Can I work remotely as an intern?",
                "info": "We support remote, hybrid, and on-site models. Remote participants receive a tech stipend and hardware. Availability depends on the specific team's infrastructure.",
                "tags": ["remote", "virtual", "online", "wfh", "location", "hybrid"]
            },
            {
                "topic": "Professional Growth",
                "query": "What training and development will I receive?",
                "info": "Our curriculum includes a 'Bootcamp Week', role-specific technical labs, and leadership workshops. You will be paired with a Senior Mentor for the duration of the program.",
                "tags": ["training", "learning", "skills", "mentorship", "development", "workshop"]
            },
            {
                "topic": "Project Scope",
                "query": "What kind of projects will I work on as an intern?",
                "info": "You will contribute to production-level code, market analysis reports, or live client deliverables. We avoid 'busy work'—your projects will have measurable business impact.",
                "tags": ["projects", "tasks", "work", "scope", "assignments"]
            }
        ]

        self.data_frame = pd.DataFrame(self.registry)
        self.master_questions = [item['query'] for item in self.registry]
        self.master_answers = [item['info'] for item in self.registry]

    def _train_similarity_model(self):
        """Converts text data into numerical vectors for comparison."""
        lowered_questions = [q.lower() for q in self.master_questions]
        self.vector_matrix = self.text_engine.fit_transform(lowered_questions)

    def _clean_input(self, text):
        return re.sub(r'[^a-zA-Z0-9\s]', '', text.lower().strip())

    def get_intelligence_match(self, user_text):
        """Uses Cosine Similarity to find the closest FAQ match."""
        cleaned = self._clean_input(user_text)
        query_vec = self.text_engine.transform([cleaned])
        scores = cosine_similarity(query_vec, self.vector_matrix)[0]

        top_idx = np.argmax(scores)
        if scores[top_idx] > 0.2:  # Similarity Threshold
            return top_idx, scores[top_idx]
        return None, 0

    def get_response(self, user_msg):
        if not user_msg.strip():
            return "How can I assist you with your internship search today?"

        # Basic Intent Mapping
        tokens = user_msg.lower()
        if any(word in tokens for word in ['hi', 'hello', 'hey']):
            return "Greetings! I am your Internship Navigator. Ask me about applications, pay, or project work."

        if any(word in tokens for word in ['thanks', 'thank you']):
            return "You're welcome! Best of luck with your career journey."

        # Perform AI Matching
        idx, confidence = self.get_intelligence_match(user_msg)

        if idx is not None:
            match = self.registry[idx]
            return f"### {match['topic']}\n\n{match['info']}\n\n*Source: Internal Program Guide*"

        # Fallback for complex queries
        return "I couldn't find a specific match for that. I can provide info on: **Applications, Requirements, Compensation, Remote Work, and Training.** Please try rephrasing!"

In [9]:
# --- INTERFACE DEPLOYMENT ---

bot = InternshipNavigator()

def process_interaction(input_text, chat_history):
    output = bot.get_response(input_text)
    chat_history.append((input_text, output))
    return chat_history, ""

with gr.Blocks(title="Internship Portal Bot", theme=gr.themes.Monochrome()) as ui:
    gr.Markdown("# 🏢 Internship Career Navigator")
    gr.Markdown("Instant, offline support for prospective interns.")

    display_box = gr.Chatbot(label="Support Log", height=450)
    input_field = gr.Textbox(placeholder="Ask about deadlines, pay, or remote work...", label="Your Question")

    input_field.submit(process_interaction, [input_field, display_box], [display_box, input_field])

    gr.Examples(
        examples=["How do I apply?", "Are internships paid?", "Is there remote work available?", "What documents do I need?"],
        inputs=input_field
    )

if __name__ == "__main__":
    ui.launch(share=True)

🛠️ System: Initializing Local Knowledge Engine...
✅ System: Navigator Ready.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e00148588f6ad498c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
